In [5]:
from pathlib import Path
import pandas as pd
import pyrosetta
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.backrub import BackrubMover
pyrosetta.init("-mute all")

PDB_PATH = Path("/mnt/c/Users/kevin/Downloads/RYR2_PDB_STRUCTURES/PDB_STRUCTURES/5go9_K167A.pdb") #change naame for models
OUTDIR = Path("/mnt/c/Users/kevin/Downloads/K167A backrub models") #change name for models
OUTDIR.mkdir(parents=True, exist_ok=True)

CHAIN = "A"
PDB_FLEX_START = 165 #residue numbers to experiment with
PDB_FLEX_END = 174

NSTRUCT = 20 #models generated
BACKRUB_TRIALS = 100 #how many do we want?

pose0 = pose_from_pdb(str(PDB_PATH))
scorefxn = pyrosetta.create_score_function("ref2015")
original_score = scorefxn(pose0)
print(f"Original score: {original_score:.3f} REU")
pdb_info = pose0.pdb_info()

FLEX_START = pdb_info.pdb2pose(CHAIN, PDB_FLEX_START) #convert pose to pdb pose
FLEX_END = pdb_info.pdb2pose(CHAIN, PDB_FLEX_END)
movemap = MoveMap()
movemap.set_bb_true_range(FLEX_START, FLEX_END)

backrub = BackrubMover()
backrub.set_movemap(movemap)
backrub.set_min_atoms(3) #change accordingly
backrub.set_max_atoms(12)

results = []

for i in range(1, NSTRUCT + 1):
    pose = pose0.clone()

    for trial in range(BACKRUB_TRIALS):
        backrub.apply(pose)

    model_score = scorefxn(pose)
    delta = model_score - original_score

    outfile = OUTDIR / f"5go9_K167A_backrub_{i:02d}.pdb" #change name for models
    pose.dump_pdb(str(outfile))

    print(f"Model {i:02d} | "f"Score: {model_score:.3f} REU | "f"Delta: {delta:.3f} REU")
    results.append({"model_number": i, "final_REU": model_score, "delta_from_original_REU": delta})

df = pd.DataFrame(results)
csv_out = OUTDIR / "K167A_backrub_scores.csv" #change name for models
df.to_csv(csv_out, index=False)

display(df)

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python312.ubuntu 2025.37+release.df75a9c48e763e52a7aa3f5dfba077f4da88dbf5 2025-09-03T12:23:30] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python312.ubuntu r408 2025.37+release.df75a9c48e df

,model_number,final_REU,delta_from_original_REU
0,1,-747.575131,530.621049
1,2,100.686115,1378.882295
2,3,-689.674254,588.521925
3,4,541.411492,1819.607671
4,5,-368.295037,909.901142
5,6,-823.439108,454.757071
6,7,-242.662072,1035.534107
7,8,212.139512,1490.335691
8,9,-1198.690706,79.505473
9,10,-858.643529,419.552651
